In [35]:
import numpy as np
from astropy.table import Table
from tqdm import tqdm
from astropy.io import fits

from astropy.coordinates import SkyCoord
from astropy.coordinates import match_coordinates_sky
from astropy import units as u

import matplotlib.pyplot as plt

In [36]:
DATA_FOLDER = '/pscratch/sd/n/nravi/GV_classification/'

In [37]:
nsa = Table.read(DATA_FOLDER + 'nsa_v1_0_1_cd_v4_cinv.fits')

## load kias table

In [41]:
kias = Table.read(DATA_FOLDER + 'kias_vagc_imaging_dr7.fits')
kias[:5]

indx,ra,dec,z,Rgal,rabsmag,urcolor,morph_class,morph_type,cd,conx,rpmag,epetR,epetRc,iso_a,iso_b,abtrue,phi_iso_deg,vdisp,vdisperr,snmedian,rdev,abdev,rexp,ab_exp,morph_err,mpmag,grcolor,devmagg,devmagr,devmagi,expmagg,expmagr,expmagi,ztype,matchdist
int64,float64,float64,float64,float64,float64,float64,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,int64,float64,float64,float64,float64,float64,float64,float64,float64,int64,float64
749,38.049133,0.224026,0.054135,160.56,-20.105,2.755,1,1.0,-0.047,0.283,15.9391,8.5,8.413,16.21,11.82,0.73,-80.52,223.27,6.29,33.68,2.434,0.807,1.536,0.833,-1,15.915,0.952,17.06,16.126,15.719,17.523,16.538,16.097,1,0.0
750,38.352528,0.212491,0.053928,159.95,-19.26,2.565,1,1.0,-0.02,0.365,16.786,5.89,5.848,8.51,7.74,0.91,-8.39,67.03,5.59,21.47,2.485,0.887,1.788,0.827,-1,16.628,0.871,17.638,16.825,16.45,18.007,17.139,16.882,1,0.0
751,38.363598,0.210654,0.05416,160.63,-20.963,2.495,2,25.88,-0.297,0.417,15.0903,13.78,13.704,25.21,10.3,0.41,-75.63,137.34,5.46,26.55,10.349,0.407,6.239,0.386,-1,15.072,0.877,15.664,15.052,14.643,15.948,15.252,14.839,1,0.0
761,54.936817,0.216794,0.201194,578.41,-21.71,2.88,1,1.0,0.034,0.323,17.3613,5.02,4.906,8.83,7.48,0.85,-67.96,245.02,13.84,13.57,2.21,0.811,1.248,0.836,-1,17.053,1.0,18.153,17.225,16.834,18.87,17.688,17.257,1,0.0
1009,54.534882,0.578615,0.128882,376.36,-20.603,2.767,1,1.0,-0.05,0.281,17.4154,4.45,4.304,8.88,6.56,0.73,69.88,197.05,10.07,15.48,1.449,0.855,1.195,0.866,-1,17.187,0.963,18.277,17.388,16.982,18.662,17.636,17.231,1,0.0


## modify kias table (old)

In [4]:
kias = Table.read(DATA_FOLDER + 'kias_vagc_imaging_dr7.dat', format='ascii')
kias[:5]

col1,col2,col3,col4,col5,col6,col7,col8,col9,col10,col11,col12,col13,col14,col15,col16,col17,col18,col19,col20,col21,col22,col23,col24,col25,col26,col27,col28,col29,col30,col31,col32,col33,col34,col35,col36
int64,float64,float64,float64,float64,float64,float64,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,int64,float64,float64,float64,float64,float64,float64,float64,float64,int64,float64
749,38.049133,0.224026,0.054135,160.56,-20.105,2.755,1,1.0,-0.047,0.283,15.9391,8.5,8.413,16.21,11.82,0.73,-80.52,223.27,6.29,33.68,2.434,0.807,1.536,0.833,-1,15.915,0.952,17.06,16.126,15.719,17.523,16.538,16.097,1,0.0
750,38.352528,0.212491,0.053928,159.95,-19.26,2.565,1,1.0,-0.02,0.365,16.786,5.89,5.848,8.51,7.74,0.91,-8.39,67.03,5.59,21.47,2.485,0.887,1.788,0.827,-1,16.628,0.871,17.638,16.825,16.45,18.007,17.139,16.882,1,0.0
751,38.363598,0.210654,0.05416,160.63,-20.963,2.495,2,25.88,-0.297,0.417,15.0903,13.78,13.704,25.21,10.3,0.41,-75.63,137.34,5.46,26.55,10.349,0.407,6.239,0.386,-1,15.072,0.877,15.664,15.052,14.643,15.948,15.252,14.839,1,0.0
761,54.936817,0.216794,0.201194,578.41,-21.71,2.88,1,1.0,0.034,0.323,17.3613,5.02,4.906,8.83,7.48,0.85,-67.96,245.02,13.84,13.57,2.21,0.811,1.248,0.836,-1,17.053,1.0,18.153,17.225,16.834,18.87,17.688,17.257,1,0.0
1009,54.534882,0.578615,0.128882,376.36,-20.603,2.767,1,1.0,-0.05,0.281,17.4154,4.45,4.304,8.88,6.56,0.73,69.88,197.05,10.07,15.48,1.449,0.855,1.195,0.866,-1,17.187,0.963,18.277,17.388,16.982,18.662,17.636,17.231,1,0.0


In [5]:
cols = ['indx', 'ra', 'dec', 'z', 'Rgal', 'rabsmag', 'urcolor', 'morph_class', 'morph_type', 'cd', 'conx', 'rpmag', 'epetR', 'epetRc', 'iso_a', 'iso_b',
        'abtrue', 'phi_iso_deg', 'vdisp', 'vdisperr', 'snmedian', 'rdev', 'abdev', 'rexp', 'ab_exp', 'morph_err', 'mpmag', 'grcolor', 'devmagg', 'devmagr', 'devmagi',
        'expmagg', 'expmagr', 'expmagi', 'ztype', 'matchdist']
len(cols)

36

In [6]:
kias.rename_columns(kias.colnames, cols)

In [7]:
kias[:5]

indx,ra,dec,z,Rgal,rabsmag,urcolor,morph_class,morph_type,cd,conx,rpmag,epetR,epetRc,iso_a,iso_b,abtrue,phi_iso_deg,vdisp,vdisperr,snmedian,rdev,abdev,rexp,ab_exp,morph_err,mpmag,grcolor,devmagg,devmagr,devmagi,expmagg,expmagr,expmagi,ztype,matchdist
int64,float64,float64,float64,float64,float64,float64,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,int64,float64,float64,float64,float64,float64,float64,float64,float64,int64,float64
749,38.049133,0.224026,0.054135,160.56,-20.105,2.755,1,1.0,-0.047,0.283,15.9391,8.5,8.413,16.21,11.82,0.73,-80.52,223.27,6.29,33.68,2.434,0.807,1.536,0.833,-1,15.915,0.952,17.06,16.126,15.719,17.523,16.538,16.097,1,0.0
750,38.352528,0.212491,0.053928,159.95,-19.26,2.565,1,1.0,-0.02,0.365,16.786,5.89,5.848,8.51,7.74,0.91,-8.39,67.03,5.59,21.47,2.485,0.887,1.788,0.827,-1,16.628,0.871,17.638,16.825,16.45,18.007,17.139,16.882,1,0.0
751,38.363598,0.210654,0.05416,160.63,-20.963,2.495,2,25.88,-0.297,0.417,15.0903,13.78,13.704,25.21,10.3,0.41,-75.63,137.34,5.46,26.55,10.349,0.407,6.239,0.386,-1,15.072,0.877,15.664,15.052,14.643,15.948,15.252,14.839,1,0.0
761,54.936817,0.216794,0.201194,578.41,-21.71,2.88,1,1.0,0.034,0.323,17.3613,5.02,4.906,8.83,7.48,0.85,-67.96,245.02,13.84,13.57,2.21,0.811,1.248,0.836,-1,17.053,1.0,18.153,17.225,16.834,18.87,17.688,17.257,1,0.0
1009,54.534882,0.578615,0.128882,376.36,-20.603,2.767,1,1.0,-0.05,0.281,17.4154,4.45,4.304,8.88,6.56,0.73,69.88,197.05,10.07,15.48,1.449,0.855,1.195,0.866,-1,17.187,0.963,18.277,17.388,16.982,18.662,17.636,17.231,1,0.0


In [ ]:
kias.write(DATA_FOLDER + 'kias_vagc_imaging_dr7.fits')

## load portsmouth table 

In [39]:
nsa_portsmouth = Table.read(DATA_FOLDER + 'NSA_v1_0_1_vflag_Portsmouth.fits')
nsa_portsmouth[:5]

IAUNAME,SUBDIR,RA,DEC,ISDSS,INED,ISIXDF,IALFALFA,IZCAT,ITWODF,MAG,Z,ZSRC,SIZE,RUN,CAMCOL,FIELD,RERUN,XPOS,YPOS,NSAID,ZDIST,SERSIC_NMGY,SERSIC_NMGY_IVAR,SERSIC_OK,SERSIC_RNMGY,SERSIC_ABSMAG,SERSIC_AMIVAR,EXTINCTION,SERSIC_KCORRECT,SERSIC_KCOEFF,SERSIC_MTOL,SERSIC_B300,SERSIC_B1000,SERSIC_METS,SERSIC_MASS,XCEN,YCEN,NPROF,PROFMEAN,PROFMEAN_IVAR,QSTOKES,USTOKES,BASTOKES,PHISTOKES,PETRO_FLUX,PETRO_FLUX_IVAR,FIBER_FLUX,FIBER_FLUX_IVAR,PETRO_BA50,PETRO_PHI50,PETRO_BA90,PETRO_PHI90,SERSIC_FLUX,SERSIC_FLUX_IVAR,SERSIC_N,SERSIC_BA,SERSIC_PHI,ASYMMETRY,CLUMPY,DFLAGS,AID,PID,DVERSION,PROFTHETA,PETRO_THETA,PETRO_TH50,PETRO_TH90,SERSIC_TH50,PLATE,FIBERID,MJD,RACAT,DECCAT,ZSDSSLINE,SURVEY,PROGRAMNAME,PLATEQUALITY,TILE,PLUG_RA,PLUG_DEC,ELPETRO_BA,ELPETRO_PHI,ELPETRO_FLUX_R,ELPETRO_FLUX_IVAR_R,ELPETRO_THETA_R,ELPETRO_TH50_R,ELPETRO_TH90_R,ELPETRO_THETA,ELPETRO_FLUX,ELPETRO_FLUX_IVAR,ELPETRO_TH50,ELPETRO_TH90,ELPETRO_APCORR_R,ELPETRO_APCORR,ELPETRO_APCORR_SELF,ELPETRO_NMGY,ELPETRO_NMGY_IVAR,ELPETRO_OK,ELPETRO_RNMGY,ELPETRO_ABSMAG,ELPETRO_AMIVAR,ELPETRO_KCORRECT,ELPETRO_KCOEFF,ELPETRO_MASS,ELPETRO_MTOL,ELPETRO_B300,ELPETRO_B1000,ELPETRO_METS,IN_DR7_LSS,u_r,g_r,NUV_r,index,imc,aimc,cd,conx1,u_r_KIAS,prmag,BPTclass,SFR,sSFR,HImass,vflag,u_r_err,g_r_err,NUV_r_err,ID,Flux_OII_3726,Flux_OII_3726_Err,AoN_OII_3726,Flux_OII_3728,Flux_OII_3728_Err,AoN_OII_3728,Flux_OIII_4363,Flux_OIII_4363_Err,AoN_OIII_4363,Flux_Hb_4861,Flux_Hb_4861_Err,AoN_Hb_4861,Flux_OIII_4958,Flux_OIII_4958_Err,Flux_OIII_5006,Flux_OIII_5006_Err,AoN_OIII_5006,Flux_NI_5197,Flux_NI_5197_Err,AoN_NI_5197,Flux_NI_5200,Flux_NI_5200_Err,AoN_NI_5200,Flux_OI_6300,Flux_OI_6300_Err,AoN_OI_6300,Flux_OI_6363,Flux_OI_6363_Err,AoN_OI_6363,Flux_NII_6547,Flux_NII_6547_Err,Flux_Ha_6562,Flux_Ha_6562_Err,AoN_Ha_6562,Flux_NII_6583,Flux_NII_6583_Err,AoN_NII_6583,Flux_SII_6716,Flux_SII_6716_Err,AoN_SII_6716,Flux_SII_6730,Flux_SII_6730_Err,AoN_SII_6730
bytes19,bytes27,float64,float64,int32,int32,int32,int32,int32,int32,float32,float32,bytes7,float32,int16,uint8,int16,bytes3,float32,float32,int32,float32,float32[7],float32[7],int16,float32[7],float32[7],float32[7],float32[7],float32[7],float32[5],float32[7],float32,float32,float32,float32,float64,float64,uint8[7],"float32[15,7]","float32[15,7]","float32[15,7]","float32[15,7]","float32[15,7]","float32[15,7]",float32[7],float32[7],float32[7],float32[7],float32,float32,float32,float32,float32[7],float32[7],float32,float32,float32,float32[7],float32[7],int32[7],int32,int32,bytes8,float32[15],float32,float32,float32,float32,int32,int16,int32,float64,float64,float32,bytes6,bytes27,bytes8,int32,float64,float64,float32,float32,float32,float32,float32,float32,float32,float32,float32[7],float32[7],float32[7],float32[7],float32,float32[7],float32[7],float32[7],float32[7],int16,float32[7],float32[7],float32[7],float32[7],float32[5],float32,float32[7],float32,float32,float32,float64,float32,float32,float32,int64,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,int64,float64,float64,float64,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64
J023211.77+001326.5,02h/p00/J023211.77+001326.5,38.04906209256324,0.22402153989646575,39438,22721,-1,-1,71327,-1,16.771198,0.054021582,sdss,0.07,109,4,23,301,1960.5631,83.361275,38979,0.054392263,1.3420134 .. 884.58374,16.215242 .. 0.0016165179,1,1.4780203 .. 964.8986,-13.908231 .. -21.045618,24.77356 .. 1073.0248,0.2271526 .. 0.04052578,-0.060923357 .. 0.029009609,1.4557369e-05 .. 1.222673e-15,0.0016189957 .. 1.23927,2.5023096e-06,0.06656253,0.024689334,2.2540354e+10,156.48558044433594,201.36697387695312,10 .. 11,0.016490662 .. 0.0,5489.7393 .. 0.0,0.21582535 .. -0.09177922,0.0076095443 .. 0.15974313,0.64479166 ..

In [13]:
nsa[:5]

IAUNAME,SUBDIR,RA,DEC,ISDSS,INED,ISIXDF,IALFALFA,IZCAT,ITWODF,MAG,Z,ZSRC,SIZE,RUN,CAMCOL,FIELD,RERUN,XPOS,YPOS,NSAID,ZDIST,SERSIC_NMGY,SERSIC_NMGY_IVAR,SERSIC_OK,SERSIC_RNMGY,SERSIC_ABSMAG,SERSIC_AMIVAR,EXTINCTION,SERSIC_KCORRECT,SERSIC_KCOEFF,SERSIC_MTOL,SERSIC_B300,SERSIC_B1000,SERSIC_METS,SERSIC_MASS,XCEN,YCEN,NPROF,PROFMEAN,PROFMEAN_IVAR,QSTOKES,USTOKES,BASTOKES,PHISTOKES,PETRO_FLUX,PETRO_FLUX_IVAR,FIBER_FLUX,FIBER_FLUX_IVAR,PETRO_BA50,PETRO_PHI50,PETRO_BA90,PETRO_PHI90,SERSIC_FLUX,SERSIC_FLUX_IVAR,SERSIC_N,SERSIC_BA,SERSIC_PHI,ASYMMETRY,CLUMPY,DFLAGS,AID,PID,DVERSION,PROFTHETA,PETRO_THETA,PETRO_TH50,PETRO_TH90,SERSIC_TH50,PLATE,FIBERID,MJD,RACAT,DECCAT,ZSDSSLINE,SURVEY,PROGRAMNAME,PLATEQUALITY,TILE,PLUG_RA,PLUG_DEC,ELPETRO_BA,ELPETRO_PHI,ELPETRO_FLUX_R,ELPETRO_FLUX_IVAR_R,ELPETRO_THETA_R,ELPETRO_TH50_R,ELPETRO_TH90_R,ELPETRO_THETA,ELPETRO_FLUX,ELPETRO_FLUX_IVAR,ELPETRO_TH50,ELPETRO_TH90,ELPETRO_APCORR_R,ELPETRO_APCORR,ELPETRO_APCORR_SELF,ELPETRO_NMGY,ELPETRO_NMGY_IVAR,ELPETRO_OK,ELPETRO_RNMGY,ELPETRO_ABSMAG,ELPETRO_AMIVAR,ELPETRO_KCORRECT,ELPETRO_KCOEFF,ELPETRO_MASS,ELPETRO_MTOL,ELPETRO_B300,ELPETRO_B1000,ELPETRO_METS,IN_DR7_LSS,g_mtot,g_m0,g_a1,g_a2,i_mtot,i_m0,i_a1,i_a2,NSA_cd,R50_i,R90_i,cinv
bytes19,bytes27,float64,float64,int32,int32,int32,int32,int32,int32,float32,float32,bytes7,float32,int16,uint8,int16,bytes3,float32,float32,int32,float32,float32[7],float32[7],int16,float32[7],float32[7],float32[7],float32[7],float32[7],float32[5],float32[7],float32,float32,float32,float32,float64,float64,uint8[7],"float32[15,7]","float32[15,7]","float32[15,7]","float32[15,7]","float32[15,7]","float32[15,7]",float32[7],float32[7],float32[7],float32[7],float32,float32,float32,float32,float32[7],float32[7],float32,float32,float32,float32[7],float32[7],int32[7],int32,int32,bytes8,float32[15],float32,float32,float32,float32,int32,int16,int32,float64,float64,float32,bytes6,bytes27,bytes8,int32,float64,float64,float32,float32,float32,float32,float32,float32,float32,float32,float32[7],float32[7],float32[7],float32[7],float32,float32[7],float32[7],float32[7],float32[7],int16,float32[7],float32[7],float32[7],float32[7],float32[5],float32,float32[7],float32,float32,float32,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64
J094651.40-010228.5,09h/m00/J094651.40-010228.5,146.714215072841,-1.0412800233313741,0,72212,37354,-1,-1,-1,15.178774,0.021222278,sdss,0.07,756,1,206,301,136.2967,1095.152,0,0.020687785,29.696056 .. 3267.6895,0.29814342 .. 0.00012172792,1,31.368013 .. 3501.2527,-15.17281 .. -20.302134,223.03555 .. 1102.6129,0.4536473 .. 0.080934174,-0.005893635 .. 0.019576037,0.00017777947 .. 5.1533486e-11,0.00020792194 .. 0.98780394,2.7473723e-05,0.31195974,0.035135116,8.898397e+09,215.39407348632812,213.4535369873047,10 .. 12,0.3149959 .. 0.0,233.37904 .. 0.0,0.062215745 .. -0.07950058,0.0006146176 .. -0.1274703,0.8828513 .. 0.7387829,0.2829979 .. -60.97547,18.203371 .. 2264.6604,1.9946122 .. 0.015697604,1.0260131 .. 561.97687,47.3397 .. 0.33574256,0.88909996,14.777527,0.80408496,17.367554,19.554192 .. 3146.785,1.0691423 .. 0.017162137,4.7761517,0.6651653,15.97821,-0.0109440535 .. 0.004525926,0.042470127 .. 0.042621203,0 .. 0,0,36,v2_1_13,0.22341923 .. 258.39,7.2478933,3.4641922,10.453795,5.882104,266,1,51630,146.71420341874853,-1.0412749124036818,0.0,sdss,legacy,good,122,146.71421,-1.0413043,0.80408496,17.367554,1144.0713,0.4582725,7.3913364,3.7061903,10.666219,7.3913364,18.787916 .. 2225.009,1.8480047 .. 0.017611798,5.8557696 .. 3.4004514,11.650886 .. 10.1460905,0.9986329,1.0687045 .. 1.001099,1.0695114 .. 1.0010818,28.532349 .. 2310.4973,0.42499655 .. 0.00024172392,1,28.661283 .. 2337.7668,-15.124495 .. -19.915525,293.50247 .. 1094.6666,-0.010805 .. 0.009306902,3.9630737e-05 .. 2.6251464e-06,6.833158e+09,0.00017553588 .. 1.14686,0.004447123,0.09061434,0.025269886,0.0,15.415104960754322,1.097097022623537,0.36327598036322273,1.4247821961562672,14.278458527287981,1.158681590197461,0.2

In [40]:
nsa_dict = {}
for i in range(len(nsa)):

    nsa_dict[nsa['IAUNAME'][i]] = i
    

In [60]:
kias_dict = {}
for i in range(len(kias)):

    kias_dict[int(kias['indx'][i])] = i

## add data to nsa table

take kias indx from portsmouth table and add to nsa table then add kias properties

In [50]:
nsa['KIAS_index'] = np.ones(len(nsa), dtype='int')*np.nan
for i in range(len(nsa_portsmouth)):

    nsa_idx = nsa_dict[nsa_portsmouth['IAUNAME'][i]]

    nsa['KIAS_index'][nsa_idx] = nsa_portsmouth['index'][i]

In [56]:
cols = kias.colnames[1:]
for col_name in cols:

    nsa['KIAS_' + col_name] = np.ones(len(nsa))*np.nan

In [65]:
for i in tqdm(range(len(nsa))):

    try:
        kias_idx = kias_dict[int(nsa['KIAS_index'][i])]
    except:
        continue
    
    for col_name in cols:

        nsa['KIAS_' + col_name][i] = kias[col_name][kias_idx]

 48%|████▊     | 307366/641409 [00:18<00:18, 17668.74it/s]/global/common/software/desi/perlmutter/desiconda/20260227-2.3.1/conda/lib/python3.13/site-packages/astropy/table/column.py:1372: UserWarning: Warning: converting a masked element to nan.
  self.data[index] = value
100%|██████████| 641409/641409 [00:34<00:00, 18449.17it/s] 


In [67]:
nsa.write(DATA_FOLDER + 'nsa_v1_0_1_cd_v4_cinv_kias.fits')